# 1. PREPROCESSING DATA (V2)

**Jurnal: Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

## Perubahan dari V1:
- Output preprocessing dibagi menjadi 2 file:
  - `data_preprocessed_ngram.csv` → dengan stopword removal (untuk N-gram/TF-IDF)
  - `data_preprocessed_indobert.csv` → tanpa stopword removal (untuk IndoBERT)

In [13]:
import pandas as pd
import numpy as np
import re
import os
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

DATA_SOURCE = '../flask_api/data/bogor_tourism_data.csv'
OUTPUT_PATH = './data/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Stopwords Indonesia
STOPWORDS_ID = {
    'yang', 'di', 'dan', 'dengan', 'ini', 'dari', 'untuk', 'adalah',
    'ke', 'pada', 'ada', 'atau', 'juga', 'bisa', 'dapat', 'akan',
    'tersebut', 'dalam', 'karena', 'oleh', 'secara', 'serta',
    'maupun', 'namun', 'tetapi', 'bahwa', 'saat', 'sangat',
    'lebih', 'sudah', 'telah', 'sedang', 'begitu', 'sehingga',
    'seperti', 'agar', 'jika', 'bila', 'hanya', 'masih', 'lagi',
    'maka', 'yaitu', 'saja', 'itu', 'nya', 'tanpa', 'kalau', 'lain',
    'diri', 'apa', 'siapa', 'mana', 'kapan', 'bagaimana', 'mengapa',
    'sebuah', 'seorang', 'suatu', 'beberapa', 'banyak', 'semua',
    'setiap', 'masing', 'para', 'tiap', 'tak', 'tidak', 'belum',
    'pun', 'pernah', 'selalu', 'terus', 'hingga', 'sampai',
    'antara', 'sebagai', 'tentang', 'terhadap', 'melalui',
    'setelah', 'sebelum', 'kemudian', 'jadi', 'harus', 'perlu',
    'ingin', 'rp', 'article', 'categories', 'tags', 'sumber', 'gambar',
    'merupakan', 'terdapat', 'memiliki', 'berbagai', 'salah', 'satu'
}

print("✅ Libraries imported!")
print(f"📝 Total stopwords: {len(STOPWORDS_ID)}")

✅ Libraries imported!
📝 Total stopwords: 99


## 1.1 Load Dataset

In [14]:
df_raw = pd.read_csv(DATA_SOURCE)

raw_info = pd.DataFrame({
    'Keterangan': ['Total Data Awal', 'Jumlah Kolom', 'Nama Kolom'],
    'Nilai': [len(df_raw), len(df_raw.columns), str(list(df_raw.columns))]
})
print("TABEL: INFO DATASET AWAL")
display(raw_info)

TABEL: INFO DATASET AWAL


,Keterangan,Nilai
0,Total Data Awal,430
1,Jumlah Kolom,7
2,Nama Kolom,"['nama', 'kategori', 'url', 'url_gambar', 'lik..."


In [15]:
# Hapus duplikat
df = df_raw.drop_duplicates(subset=['nama'], keep='first').reset_index(drop=True)

# Distribusi kategori
cat_dist = df['kategori'].value_counts().reset_index()
cat_dist.columns = ['Kategori', 'Jumlah']
cat_dist['Persentase'] = (cat_dist['Jumlah'] / len(df) * 100).round(2).astype(str) + '%'

print("TABEL: DISTRIBUSI KATEGORI")
display(cat_dist)

TABEL: DISTRIBUSI KATEGORI


,Kategori,Jumlah,Persentase
0,Arena,82,27.7%
1,Alam,82,27.7%
2,Rekreasi,62,20.95%
3,Kuliner,38,12.84%
4,Olahraga,11,3.72%
5,Seni Budaya,11,3.72%
6,Belanja,10,3.38%


## 1.2 Text Preprocessing

In [16]:
def preprocess_basic(text):
    """Preprocessing dasar TANPA stopword removal (untuk IndoBERT)"""
    if pd.isna(text) or text == '':
        return ''
    text = str(text).lower()                      # Case folding
    text = re.sub(r'http\S+|www\S+', '', text)    # Remove URLs
    text = re.sub(r'\d+', '', text)               # Remove numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)      # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()      # Remove extra spaces
    return text

def remove_stopwords(text):
    """Hapus stopword dari teks (untuk N-gram)"""
    if not text:
        return ''
    words = text.split()
    filtered = [w for w in words if w.lower() not in STOPWORDS_ID]
    return ' '.join(filtered)

# Apply preprocessing
df['deskripsi_clean'] = df['deskripsi'].apply(preprocess_basic)  # Untuk IndoBERT
df['deskripsi_ngram'] = df['deskripsi_clean'].apply(remove_stopwords)  # Untuk N-gram

preprocess_steps = pd.DataFrame({
    'Langkah': ['1. Case Folding', '2. Remove URLs', '3. Remove Numbers', 
                '4. Remove Special Chars', '5. Remove Extra Spaces', '6. Remove Stopwords (N-gram only)'],
    'Deskripsi': ['Konversi ke huruf kecil', 'Hapus URL (http/www)', 'Hapus angka', 
                  'Hapus karakter spesial', 'Hapus spasi berlebih', 'Hapus stopword Indonesia']
})
print("TABEL: LANGKAH PREPROCESSING")
display(preprocess_steps)

TABEL: LANGKAH PREPROCESSING


,Langkah,Deskripsi
0,1. Case Folding,Konversi ke huruf kecil
1,2. Remove URLs,Hapus URL (http/www)
2,3. Remove Numbers,Hapus angka
3,4. Remove Special Chars,Hapus karakter spesial
4,5. Remove Extra Spaces,Hapus spasi berlebih
5,6. Remove Stopwords (N-gram only),Hapus stopword Indonesia


In [17]:
# Before-After Table
before_after = []
for i in range(min(5, len(df))):
    before_after.append({
        'Nama Wisata': df.iloc[i]['nama'],
        'Deskripsi Asli (80 char)': str(df.iloc[i]['deskripsi'])[:80] + '...',
        'IndoBERT (80 char)': df.iloc[i]['deskripsi_clean'][:80] + '...',
        'N-gram (80 char)': df.iloc[i]['deskripsi_ngram'][:80] + '...'
    })

ba_df = pd.DataFrame(before_after)
print("TABEL: HASIL PREPROCESSING (Before - After)")
display(ba_df)

TABEL: HASIL PREPROCESSING (Before - After)


,Nama Wisata,Deskripsi Asli (80 char),IndoBERT (80 char),N-gram (80 char)
0,Curug Ciampea,"Curug Ciampea Bogor, adalah salah – satu peson...",curug ciampea bogor adalah salah satu pesona a...,curug ciampea bogor pesona alam berada kaki gu...
1,Bukit Cirimpak,Bukit Cirimpak salah satu camping ground yang ...,bukit cirimpak salah satu camping ground yang ...,bukit cirimpak camping ground kota bogor jawa ...
2,Lembah Tepus,Lembah Tepus adalah destinasi wisata berupa su...,lembah tepus adalah destinasi wisata berupa su...,lembah tepus destinasi wisata berupa sungai ai...
3,Sun Water Park Kahuripan,Sun Water Park Kahuripan menawarkan kawasan wi...,sun water park kahuripan menawarkan kawasan wi...,sun water park kahuripan menawarkan kawasan wi...
4,Lembah Pinus Camp & Café,Lembah Pinus Camp & Cafe merupakan salah satu ...,lembah pinus camp cafe merupakan salah satu wi...,lembah pinus camp cafe wisata terletak kabupat...


## 1.3 Simpan Data

In [ ]:
# File untuk IndoBERT (tanpa stopword removal)
# Perbaikan: Gunakan 'likes' bukan 'likasi'
df_indobert = df[['nama', 'kategori', 'url', 'url_gambar', 'likes', 'deskripsi', 'deskripsi_clean']].copy()
df_indobert.to_csv(f'{OUTPUT_PATH}data_preprocessed_indobert.csv', index=False, encoding='utf-8-sig')

# File untuk N-gram (dengan stopword removal)
df_ngram = df[['nama', 'kategori', 'url', 'url_gambar', 'likes', 'deskripsi', 'deskripsi_clean', 'deskripsi_ngram']].copy()
df_ngram.to_csv(f'{OUTPUT_PATH}data_preprocessed_ngram.csv', index=False, encoding='utf-8-sig')

# File legacy untuk kompatibilitas
df.to_csv(f'{OUTPUT_PATH}data_preprocessed.csv', index=False, encoding='utf-8-sig')

# Summary
saved = pd.DataFrame({
    'File': ['data_preprocessed_indobert.csv', 'data_preprocessed_ngram.csv', 'data_preprocessed.csv'],
    'Deskripsi': ['Untuk IndoBERT (tanpa stopword)', 'Untuk N-gram (dengan stopword removal)', 'Legacy (kompatibilitas)'],
    'Rows': [len(df_indobert), len(df_ngram), len(df)]
})
print("TABEL: FILE SAVED")
display(saved)

print("\n✅ PREPROCESSING SELESAI!")

TABEL: FILE SAVED


,File,Deskripsi,Rows
0,data_preprocessed_indobert.csv,Untuk IndoBERT (tanpa stopword),296
1,data_preprocessed_ngram.csv,Untuk N-gram (dengan stopword removal),296
2,data_preprocessed.csv,Legacy (kompatibilitas),296



✅ PREPROCESSING SELESAI!


: 